In [ ]:
from Bio import SeqIO, AlignIO, Phylo, Entrez
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Blast import NCBIWWW, NCBIXML
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor

import csv
import os
import time
import re
import subprocess

# ---- PATHS ----
pasta_inputs = r"C:\Users\dbran\Desktop\BioInformática\Labs"
ficheiros_fasta = [
    os.path.join(pasta_inputs, "14296266.fasta"),
    os.path.join(pasta_inputs, "14296280.fasta"),
    os.path.join(pasta_inputs, "14296281.fasta"),
    os.path.join(pasta_inputs, "14296289.fasta"),
]

pasta_resultados = os.path.join(pasta_inputs, "resultados") 
os.makedirs(pasta_resultados, exist_ok=True)

ficheiro_proteinas = os.path.join(pasta_resultados, "nossos_genes_traduzidos.faa")

pasta_blast = os.path.join(pasta_resultados, "blastp_xml")
os.makedirs(pasta_blast, exist_ok=True)

muscle_exe = r"C:\Users\dbran\Desktop\BioInformática\Labs\muscle\muscle.exe"

# sanity check
for f in ficheiros_fasta:
    print(f, "OK" if os.path.exists(f) else "MISSING")
print("Resultados:", pasta_resultados)
print("MUSCLE existe?", os.path.exists(muscle_exe))


C:\Users\dbran\Desktop\BioInformática\Labs\14296266.fasta OK
C:\Users\dbran\Desktop\BioInformática\Labs\14296280.fasta OK
C:\Users\dbran\Desktop\BioInformática\Labs\14296281.fasta OK
C:\Users\dbran\Desktop\BioInformática\Labs\14296289.fasta OK
Resultados: C:\Users\dbran\Desktop\BioInformática\Labs\resultados
MUSCLE existe? True


In [ ]:
#DNA
def e_dna(sequencia: str) -> bool:
    return set(sequencia.upper()) <= set("ACGTN")

registos_dna = []
for caminho_fasta in ficheiros_fasta:
    registo = SeqIO.read(caminho_fasta, "fasta")
    seq_str = str(registo.seq).upper().replace(" ", "").replace("\n", "")
    if not e_dna(seq_str):
        raise ValueError(f"{caminho_fasta} não parece DNA (há caracteres fora de A/C/G/T/N).")
    registo.seq = Seq(seq_str)
    registos_dna.append(registo)

# resumo rápido
[(r.id, len(r.seq), r.description) for r in registos_dna]


[('NC_019914.1:16482-17621',
  1140,
  'NC_019914.1:16482-17621 Staphylococcus phage StB27, complete genome'),
 ('NC_019914.1:29102-30352',
  1251,
  'NC_019914.1:29102-30352 Staphylococcus phage StB27, complete genome'),
 ('NC_019914.1:30363-32237',
  1875,
  'NC_019914.1:30363-32237 Staphylococcus phage StB27, complete genome'),
 ('NC_019914.1:38194-39930',
  1737,
  'NC_019914.1:38194-39930 Staphylococcus phage StB27, complete genome')]

In [5]:
#Traduzir ORF
def melhor_traducao_orf_6frames(dna: Seq, min_aa=60):
    dna = dna.upper()
    candidatos = []

    for sentido, seq in [("+", dna), ("-", dna.reverse_complement())]:
        for frame in range(3):
            prot = seq[frame:].translate(to_stop=False)
            partes = str(prot).split("*")
            for p in partes:
                if "M" not in p:
                    continue
                p2 = p[p.find("M"):]
                if len(p2) >= min_aa:
                    candidatos.append((len(p2), sentido, frame, p2))

    if not candidatos:
        return None, None, None

    candidatos.sort(reverse=True, key=lambda x: x[0])
    _, melhor_sentido, melhor_frame, melhor_aa = candidatos[0]
    return melhor_aa, melhor_sentido, melhor_frame

registos_proteina = []
info_orf = []

for r in registos_dna:
    aa, sentido, frame = melhor_traducao_orf_6frames(r.seq, min_aa=60)
    if aa is None:
        aa = str(r.seq.translate(to_stop=True))
        sentido, frame = "+", 0

    registos_proteina.append(
        SeqRecord(Seq(aa), id=r.id, description=f"traducao_melhor_orf sentido={sentido} frame={frame}")
    )
    info_orf.append((r.id, len(r.seq), len(aa), sentido, frame))

info_orf

c:\Users\dbran\anaconda3\Lib\site-packages\Bio\Seq.py:2877: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


[('NC_019914.1:16482-17621', 1140, 379, '+', 0),
 ('NC_019914.1:29102-30352', 1251, 416, '+', 0),
 ('NC_019914.1:30363-32237', 1875, 624, '+', 0),
 ('NC_019914.1:38194-39930', 1737, 578, '+', 0)]

In [6]:
#Guardar Proteinas Traduzidas
SeqIO.write(registos_proteina, ficheiro_proteinas, "fasta")
ficheiro_proteinas, [(p.id, len(p.seq)) for p in registos_proteina]


('C:\\Users\\dbran\\Desktop\\BioInformática\\Labs\\resultados\\nossos_genes_traduzidos.faa',
 [('NC_019914.1:16482-17621', 379),
  ('NC_019914.1:29102-30352', 416),
  ('NC_019914.1:30363-32237', 624),
  ('NC_019914.1:38194-39930', 578)])

In [8]:
#BlastP
def executar_blastp(registo_proteina, base_dados="nr", max_hits=10):
    handle = NCBIWWW.qblast("blastp", base_dados, registo_proteina.format("fasta"), hitlist_size=max_hits)
    xml_texto = handle.read()
    handle.close()
    return xml_texto

def guardar_xml_blast(xml_texto, caminho_xml):
    with open(caminho_xml, "w", encoding="utf-8") as f:
        f.write(xml_texto)

def ler_top_hits(caminho_xml, n=5):
    with open(caminho_xml) as handle:
        registo_blast = NCBIXML.read(handle)

    hits = []
    for alinhamento in registo_blast.alignments[:n]:
        hsp = alinhamento.hsps[0]
        hits.append({
            "hit_def": alinhamento.hit_def,
            "hit_id": getattr(alinhamento, "hit_id", None),
            "accession": getattr(alinhamento, "accession", None),  
            "hit_length": getattr(alinhamento, "length", None),
            "tamanho_alinhamento": hsp.align_length,
            "identidades": hsp.identities,
            "percent_id": 100.0 * hsp.identities / hsp.align_length if hsp.align_length else None,
            "evalue": hsp.expect
        })
    return hits


resumo_blast = {}

for registo in registos_proteina:
    safe_id = registo.id.replace(":", "_").replace("-", "_")
    caminho_xml = os.path.join(pasta_blast, f"{safe_id}_blastp.xml")

    if not os.path.exists(caminho_xml):
        xml = executar_blastp(registo, base_dados="nr", max_hits=10)
        guardar_xml_blast(xml, caminho_xml)
        time.sleep(15)

    resumo_blast[registo.id] = ler_top_hits(caminho_xml, n=10)

resultados_blast = resumo_blast
list(resultados_blast.keys())


['NC_019914.1:16482-17621',
 'NC_019914.1:29102-30352',
 'NC_019914.1:30363-32237',
 'NC_019914.1:38194-39930']

In [7]:
out_csv = os.path.join(pasta_resultados, "resumo_genes_blast.csv")

with open(out_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["gene_id", "len_nt", "len_aa", "top_hit_accession", "top_hit_def", "top_hit_evalue", "top_hit_percent_id"])
    for (gene_id, len_nt, len_aa, sentido, frame) in info_orf:
        hits = resultados_blast.get(gene_id, [])
        if hits:
            h0 = hits[0]
            w.writerow([gene_id, len_nt, len_aa, h0.get("accession"), h0.get("hit_def"), h0.get("evalue"), h0.get("percent_id")])
        else:
            w.writerow([gene_id, len_nt, len_aa, None, None, None, None])

out_csv


'C:\\Users\\dbran\\Desktop\\BioInformática\\Labs\\resultados\\resumo_genes_blast.csv'

In [ ]:
#Entrez
Entrez.email = "dbrandao2004@gmail.com"

top_hits_por_gene = 10
evalue_max = 1e-20
min_percent_id = 30.0

fasta_saida = os.path.join(pasta_resultados, "para_filogenia_com_homologos.faa")

def extrair_accession_de_hitdef(hit_def: str):
    m = re.search(r"\b[A-Z]{1,4}\d{4,}\.\d+\b", hit_def)
    return m.group(0) if m else None

def fetch_proteinas_fasta(accessions):
    with Entrez.efetch(db="protein", id=",".join(accessions), rettype="fasta", retmode="text") as h:
        return list(SeqIO.parse(h, "fasta"))


meus_recs = list(SeqIO.parse(ficheiro_proteinas, "fasta"))


accessions_por_gene = {}
todos_accessions = []

for gene_id, hits in resultados_blast.items():
    escolhidos = []
    for h in hits:
        ev = h.get("evalue", None)
        pid = h.get("percent_id", None)

        if ev is not None and ev > evalue_max:
            continue
        if pid is not None and pid < min_percent_id:
            continue

        acc = h.get("accession") or extrair_accession_de_hitdef(h.get("hit_def", ""))
        if not acc:
            continue

        if acc not in escolhidos:
            escolhidos.append(acc)

        if len(escolhidos) >= top_hits_por_gene:
            break

    accessions_por_gene[gene_id] = escolhidos
    todos_accessions.extend(escolhidos)


seen = set()
todos_accessions = [a for a in todos_accessions if not (a in seen or seen.add(a))]


homologos_recs = []
batch = 50
for i in range(0, len(todos_accessions), batch):
    bloco = todos_accessions[i:i+batch]
    homologos_recs.extend(fetch_proteinas_fasta(bloco))
    time.sleep(0.4)


def encurtar(txt, n=60):
    txt = re.sub(r"\s+", " ", txt).strip()
    return (txt[:n] + "...") if len(txt) > n else txt

homologos_final = []
for gene_id, accs in accessions_por_gene.items():
    for acc in accs:
        rec = None
        for r in homologos_recs:
            if r.id == acc or r.id.startswith(acc):
                rec = r
                break
        if rec is None:
            continue

        rec2 = rec[:]
        rec2.id = f"{gene_id}|{rec.id}"
        rec2.description = encurtar(rec.description)
        homologos_final.append(rec2)

todos_recs = []
for r in meus_recs:
    r2 = r[:]
    r2.id = f"MEU|{r.id}"
    r2.description = encurtar(r.description)
    todos_recs.append(r2)

todos_recs.extend(homologos_final)

SeqIO.write(todos_recs, fasta_saida, "fasta")
print("Guardado:", fasta_saida)
print("Total seqs:", len(todos_recs))


Guardado: C:\Users\dbran\Desktop\BioInformática\Labs\resultados\para_filogenia_com_homologos.faa
Total seqs: 44


In [ ]:
#Features

def parse_intervalo(gene_id):
   
    m = re.match(r"^([^:]+):(\d+)-(\d+)$", gene_id)
    if not m:
        return None
    return m.group(1), int(m.group(2)), int(m.group(3))

def fetch_genbank_genoma(accession, pasta_out):
    os.makedirs(pasta_out, exist_ok=True)
    out = os.path.join(pasta_out, f"{accession}.gb")
    if os.path.exists(out):
        return out
    with Entrez.efetch(db="nucleotide", id=accession, rettype="gb", retmode="text") as h:
        txt = h.read()
    with open(out, "w", encoding="utf-8") as f:
        f.write(txt)
    time.sleep(0.4)
    return out


genoma_acc = parse_intervalo(registos_dna[0].id)[0]
gb_path = fetch_genbank_genoma(genoma_acc, os.path.join(pasta_resultados, "genbank"))
record = SeqIO.read(gb_path, "genbank")

gb_path, record.id, record.description


('C:\\Users\\dbran\\Desktop\\BioInformática\\Labs\\resultados\\genbank\\NC_019914.1.gb',
 'NC_019914.1',
 'Staphylococcus phage StB27, complete genome')

In [ ]:
def features_que_intersectam(record, inicio, fim):
    feats = []
    for ft in record.features:
        if ft.type != "CDS":
            continue
        if ft.location is None:
            continue
        s = int(ft.location.start)
        e = int(ft.location.end)
       
        if not (e < inicio or s > fim):
            feats.append((s, e, ft.qualifiers))
    return feats

for gene_id in [r.id for r in registos_dna]:
    acc, ini, fim = parse_intervalo(gene_id)
    feats = features_que_intersectam(record, ini, fim)
    print("\n==", gene_id, "==")
    for s, e, q in feats[:3]:  
        print("CDS", s, "-", e,
              "| gene:", q.get("gene", ["?"])[0],
              "| product:", q.get("product", ["?"])[0],
              "| protein_id:", q.get("protein_id", ["?"])[0])



== NC_019914.1:16482-17621 ==
CDS 16481 - 17621 | gene: ? | product: terminase large subunit | protein_id: YP_007236598.1
CDS 17621 - 19073 | gene: ? | product: portal protein | protein_id: YP_007236599.1

== NC_019914.1:29102-30352 ==
CDS 29101 - 30352 | gene: ? | product: tail protein with endopeptidase domain | protein_id: YP_007236612.1

== NC_019914.1:30363-32237 ==
CDS 30362 - 32237 | gene: ? | product: zinc carboxypeptidase | protein_id: YP_007236613.1

== NC_019914.1:38194-39930 ==
CDS 38193 - 39930 | gene: ? | product: endolysin | protein_id: YP_007236621.1


In [11]:
out_feat = os.path.join(pasta_resultados, "features_genbank_por_gene.csv")

with open(out_feat, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["gene_id", "cds_start", "cds_end", "gene", "product", "protein_id"])

    for gene_id in [r.id for r in registos_dna]:
        acc, ini, fim = parse_intervalo(gene_id)
        feats = features_que_intersectam(record, ini, fim)
        for s, e, q in feats:
            w.writerow([
                gene_id,
                s, e,
                q.get("gene", ["?"])[0],
                q.get("product", ["?"])[0],
                q.get("protein_id", ["?"])[0]
            ])

out_feat


'C:\\Users\\dbran\\Desktop\\BioInformática\\Labs\\resultados\\features_genbank_por_gene.csv'

In [31]:
# Alinhar + Tree
ficheiro_alinhamento = os.path.join(pasta_resultados, "para_filogenia_com_homologos.aln.fasta")

subprocess.run([muscle_exe, "-in", fasta_saida, "-out", ficheiro_alinhamento], check=True)

alinhamento = AlignIO.read(ficheiro_alinhamento, "fasta")
print("Alinhamento:", alinhamento.get_alignment_length(), "posições |", len(alinhamento), "seqs")

# árvore NJ
calculadora = DistanceCalculator("identity")
matriz_distancias = calculadora.get_distance(alinhamento)

construtor = DistanceTreeConstructor()
arvore = construtor.nj(matriz_distancias)

ficheiro_arvore = os.path.join(pasta_resultados, "arvore_com_homologos.nwk")
Phylo.write(arvore, ficheiro_arvore, "newick")

Phylo.draw_ascii(arvore)
ficheiro_arvore


Alinhamento: 697 posições | 44 seqs
                                 , NC_019914.1:38194-39930|WP_458622396.1
                                 |
                                 | NC_019914.1:38194-39930|XOQ91196.1
                                 |
                                 , NC_019914.1:38194-39930|WP_434179235.1
                                 |
                                 | NC_019914.1:38194-39930|WP_415401313.1
                                 |
                                 | NC_019914.1:38194-39930|WP_170076343.1
                                 |
                                 | NC_019914.1:38194-39930|WP_053028142.1
                                 |
                                 , NC_019914.1:38194-39930|YP_007236621.1
                                 |
                _________________| MEU|NC_019914.1:38194-39930
               |                 |
               |                 | NC_019914.1:38194-39930|WP_207025504.1
               |                 

'C:\\Users\\dbran\\Desktop\\BioInformática\\Labs\\resultados\\arvore_com_homologos.nwk'